In [ ]:
````xml
<VSCode.Cell language="markdown">
# LSTM ESG Trend Forecaster

**Purpose**: Predict 12-month ESG trajectory using time series forecasting

**Model**: LSTM (Long Short-Term Memory) neural network

**Input**: Last 6 years of ESG scores → **Output**: Next 6 years forecast (interpreted as 12-month trend)

**Use Case**: Detect if company ESG is IMPROVING, STABLE, or DECLINING

---

## Requirements
- Google Colab (GPU runtime recommended)
- Dataset: `company_esg_financial_dataset.csv` (11,000 rows with Year column)
- Training time: ~10-15 minutes (GPU), ~30-40 minutes (CPU)
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Section 1: Setup and Install Dependencies
</VSCode.Cell>
<VSCode.Cell language="python">
!pip install tensorflow keras scikit-learn matplotlib pandas numpy joblib -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
import joblib
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")
print(f"TensorFlow GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Section 2: Load and Prepare Time Series Data

We'll convert the company-level data into time series sequences suitable for LSTM training.
</VSCode.Cell>
<VSCode.Cell language="python">
# Upload dataset
from google.colab import files
print("📁 Please upload: company_esg_financial_dataset.csv")
uploaded = files.upload()

# Load data
df = pd.read_csv('company_esg_financial_dataset.csv')
print(f"\n✅ Dataset loaded: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst rows:\n{df.head()}")
</VSCode.Cell>
<VSCode.Cell language="python">
# Sort by Company and Year
df = df.sort_values(['CompanyID', 'Year'])

# Group by Company and Year - aggregate ESG scores
df_ts = df.groupby(['CompanyID', 'Year']).agg({
    'ESG_Overall': 'mean',
    'ESG_Environmental': 'mean',
    'ESG_Social': 'mean',
    'ESG_Governance': 'mean'
}).reset_index()

print(f"\n{'='*60}")
print("TIME SERIES DATA SUMMARY")
print(f"{'='*60}")
print(f"Shape: {df_ts.shape}")
print(f"Companies: {df_ts['CompanyID'].nunique()}")
print(f"Year range: {df_ts['Year'].min()} to {df_ts['Year'].max()}")
print(f"\nSample time series (first company):\n")
print(df_ts[df_ts['CompanyID'] == df_ts['CompanyID'].iloc[0]].head(10))
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Section 3: Create Sliding Window Sequences

**Approach**: Use last 6 years to predict next 6 years

**Example**: `[65, 67, 68, 70, 72, 75]` → `[76, 78, 79, 80, 81, 82]`
</VSCode.Cell>
<VSCode.Cell language="python">
def create_sequences(data, company_col='CompanyID', target_col='ESG_Overall', 
                     seq_length=6, forecast_length=6):
    """
    Create sequences: [t-5, t-4, ..., t] → [t+1, t+2, ..., t+6]
    
    Args:
        data: DataFrame with company ESG time series
        seq_length: Number of historical years to use (input)
        forecast_length: Number of future years to predict (output)
    
    Returns:
        X: Array of input sequences (samples, seq_length)
        y: Array of target sequences (samples, forecast_length)
    """
    X, y = [], []
    
    for company_id in data[company_col].unique():
        company_data = data[data[company_col] == company_id][target_col].values
        
        # Need at least seq_length + forecast_length data points
        if len(company_data) < seq_length + forecast_length:
            continue
        
        # Create sliding windows
        for i in range(len(company_data) - seq_length - forecast_length + 1):
            X.append(company_data[i:i+seq_length])
            y.append(company_data[i+seq_length:i+seq_length+forecast_length])
    
    return np.array(X), np.array(y)

# Create sequences
seq_length = 6  # Use last 6 years
forecast_length = 6  # Predict next 6 years

print("🔄 Creating sequences...")
X, y = create_sequences(df_ts, seq_length=seq_length, forecast_length=forecast_length)

print(f"\n{'='*60}")
print("SEQUENCES CREATED")
print(f"{'='*60}")
print(f"X shape: {X.shape} - (samples, sequence_length)")
print(f"y shape: {y.shape} - (samples, forecast_length)")
print(f"\nExample sequence:")
print(f"Input (last 6 years):  {X[0]}")
print(f"Target (next 6 years): {y[0]}")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Section 4: Normalize and Split Data

LSTM works best with normalized data (0-1 range).
</VSCode.Cell>
<VSCode.Cell language="python">
# Normalize to 0-1 range
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

# Reshape for LSTM: (samples, timesteps, features)
X_scaled = X_scaled.reshape((X_scaled.shape[0], X_scaled.shape[1], 1))

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_scaled, 
    test_size=0.2, 
    random_state=42
)

print(f"\n{'='*60}")
print("DATA SPLIT")
print(f"{'='*60}")
print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")
print(f"\nScaling:")
print(f"Original range: [{X.min():.1f}, {X.max():.1f}]")
print(f"Scaled range:   [{X_scaled.min():.3f}, {X_scaled.max():.3f}]")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Section 5: Build LSTM Model

**Architecture**:
- LSTM Layer 1: 64 units (returns sequences)
- Dropout: 20% (prevents overfitting)
- LSTM Layer 2: 32 units (final encoding)
- Dropout: 20%
- Dense Output: 6 neurons (6-year forecast)
</VSCode.Cell>
<VSCode.Cell language="python">
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(seq_length, 1)),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(forecast_length)  # Predict 6 future values
])

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

print("\n" + "="*60)
print("MODEL ARCHITECTURE")
print("="*60)
model.summary()
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Section 6: Train Model

**Training Configuration**:
- Epochs: 50 (with early stopping)
- Batch size: 32
- Validation split: 20%
- Early stopping: Patience 10 epochs

**Expected time**: 10-15 minutes (GPU), 30-40 minutes (CPU)
</VSCode.Cell>
<VSCode.Cell language="python">
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

print("\n" + "="*60)
print("TRAINING LSTM MODEL")
print("="*60)
print("This may take 10-15 minutes...\n")

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

print("\n✅ Training complete!")
</VSCode.Cell>
<VSCode.Cell language="python">
# Plot training history
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Val Loss', linewidth=2)
plt.title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Train MAE', linewidth=2)
plt.plot(history.history['val_mae'], label='Val MAE', linewidth=2)
plt.title('Model MAE Over Epochs', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Section 7: Evaluate Model

Test the model on unseen data and calculate performance metrics.
</VSCode.Cell>
<VSCode.Cell language="python">
# Predict on test set
y_pred_scaled = model.predict(X_test)

# Inverse transform to original scale
y_test_original = scaler_y.inverse_transform(y_test)
y_pred_original = scaler_y.inverse_transform(y_pred_scaled)

# Calculate metrics
mae = np.mean(np.abs(y_test_original - y_pred_original))
rmse = np.sqrt(np.mean((y_test_original - y_pred_original)**2))
mape = np.mean(np.abs((y_test_original - y_pred_original) / y_test_original)) * 100

print(f"\n{'='*60}")
print("LSTM FORECASTING PERFORMANCE")
print(f"{'='*60}")
print(f"MAE:   {mae:.2f} points (average error per prediction)")
print(f"RMSE:  {rmse:.2f} points (penalizes large errors)")
print(f"MAPE:  {mape:.1f}% (percentage error)")
print(f"{'='*60}")
print(f"\nInterpretation:")
print(f"  MAE < 5:  ✅ Excellent")
print(f"  MAE 5-10: ⚠️  Good")
print(f"  MAE > 10: ❌ Needs improvement")
</VSCode.Cell>
<VSCode.Cell language="python">
# Visualize predictions
num_samples = min(5, len(y_test_original))
fig, axes = plt.subplots(num_samples, 1, figsize=(12, 3*num_samples))

if num_samples == 1:
    axes = [axes]

for i in range(num_samples):
    ax = axes[i]
    years = list(range(1, 7))
    
    ax.plot(years, y_test_original[i], 'o-', label='Actual', 
            linewidth=2, markersize=8, color='#2E86AB')
    ax.plot(years, y_pred_original[i], 's-', label='Predicted', 
            linewidth=2, markersize=8, color='#A23B72')
    
    ax.set_xlabel('Year', fontsize=11)
    ax.set_ylabel('ESG Score', fontsize=11)
    ax.set_title(f'Sample {i+1}: 6-Year ESG Forecast', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 100])

plt.tight_layout()
plt.show()
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Section 8: Trend Detection Function

Convert 6-year forecast into actionable trend classification.
</VSCode.Cell>
<VSCode.Cell language="python">
def detect_trend(forecast):
    """
    Determine if ESG is IMPROVING, STABLE, or DECLINING
    
    Args:
        forecast: Array of future ESG scores (length 6)
    
    Returns:
        trend: "IMPROVING", "STABLE", or "DECLINING"
        change_pct: Percentage change from start to end
    """
    start = forecast[:2].mean()  # Average of first 2 years
    end = forecast[-2:].mean()   # Average of last 2 years
    change_pct = ((end - start) / start) * 100
    
    if change_pct > 5:
        trend = "IMPROVING"
    elif change_pct < -5:
        trend = "DECLINING"
    else:
        trend = "STABLE"
    
    return trend, change_pct

# Test on predictions
print(f"\n{'='*60}")
print("TREND ANALYSIS (Sample Forecasts)")
print(f"{'='*60}")

correct_trends = 0
total_samples = min(10, len(y_pred_original))

for i in range(total_samples):
    forecast = y_pred_original[i]
    actual = y_test_original[i]
    
    pred_trend, pred_change = detect_trend(forecast)
    actual_trend, actual_change = detect_trend(actual)
    
    match = pred_trend == actual_trend
    if match:
        correct_trends += 1
    
    print(f"\nSample {i+1}:")
    print(f"  Predicted: {pred_trend:10s} ({pred_change:+.1f}%)")
    print(f"  Actual:    {actual_trend:10s} ({actual_change:+.1f}%)")
    print(f"  Match: {'✅' if match else '❌'}")

trend_accuracy = (correct_trends / total_samples) * 100
print(f"\n{'='*60}")
print(f"Trend Classification Accuracy: {trend_accuracy:.1f}%")
print(f"{'='*60}")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Section 9: Save Model and Scalers

Save the trained model and scalers for deployment.
</VSCode.Cell>
<VSCode.Cell language="python">
# Save model
model.save('lstm_trend_forecaster.h5')
print("\n✅ Model saved: lstm_trend_forecaster.h5")

# Save scalers (need both for predictions)
joblib.dump(scaler_X, 'lstm_scaler_X.pkl')
joblib.dump(scaler_y, 'lstm_scaler_y.pkl')
print("✅ Scalers saved: lstm_scaler_X.pkl, lstm_scaler_y.pkl")

# Save metadata
metadata = {
    'mae': float(mae),
    'rmse': float(rmse),
    'mape': float(mape),
    'seq_length': seq_length,
    'forecast_length': forecast_length,
    'train_samples': len(X_train),
    'test_samples': len(X_test),
    'trend_accuracy': float(trend_accuracy)
}

import json
with open('lstm_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("✅ Metadata saved: lstm_metadata.json")

# Download from Colab
try:
    from google.colab import files
    files.download('lstm_trend_forecaster.h5')
    files.download('lstm_scaler_X.pkl')
    files.download('lstm_scaler_y.pkl')
    files.download('lstm_metadata.json')
    print("\n✅ Files ready for download!")
except:
    print("\n⚠️  Not in Colab - files saved locally")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Section 10: Test Prediction Function

Define the prediction function that will be used in production.
</VSCode.Cell>
<VSCode.Cell language="python">
def predict_trend(historical_esg_scores):
    """
    Predict future ESG trend from historical scores
    
    Args:
        historical_esg_scores: List of last 6 ESG scores (e.g., [60, 62, 65, 68, 70, 72])
    
    Returns:
        {
            'forecast': array of 6 future scores,
            'trend': 'IMPROVING'/'STABLE'/'DECLINING',
            'change_pct': percentage change,
            'confidence_mae': model MAE
        }
    """
    # Prepare input
    X_input = np.array(historical_esg_scores).reshape(1, -1)
    X_scaled = scaler_X.transform(X_input)
    X_scaled = X_scaled.reshape(1, seq_length, 1)
    
    # Predict
    y_pred_scaled = model.predict(X_scaled, verbose=0)
    forecast = scaler_y.inverse_transform(y_pred_scaled)[0]
    
    # Detect trend
    trend, change_pct = detect_trend(forecast)
    
    return {
        'forecast': forecast.round(1).tolist(),
        'trend': trend,
        'change_pct': round(change_pct, 1),
        'confidence_mae': round(mae, 2)
    }

# Test with sample data
print(f"\n{'='*60}")
print("TEST PREDICTIONS")
print(f"{'='*60}")

test_cases = [
    ([60, 62, 65, 68, 70, 72], "Improving trend"),
    ([75, 73, 70, 68, 65, 62], "Declining trend"),
    ([65, 66, 65, 67, 66, 65], "Stable trend")
]

for test_history, description in test_cases:
    result = predict_trend(test_history)
    
    print(f"\n{description}:")
    print(f"  Input:    {test_history}")
    print(f"  Forecast: {result['forecast']}")
    print(f"  Trend:    {result['trend']}")
    print(f"  Change:   {result['change_pct']:+.1f}%")
    print(f"  MAE:      {result['confidence_mae']} points")

print(f"\n{'='*60}")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Section 11: Final Summary

Review model performance and deployment instructions.
</VSCode.Cell>
<VSCode.Cell language="python">
print(f"\n{'='*70}")
print("LSTM ESG TREND FORECASTER - TRAINING COMPLETE")
print(f"{'='*70}")
print(f"\n📊 Model Performance:")
print(f"   MAE:  {mae:.2f} points")
print(f"   RMSE: {rmse:.2f} points")
print(f"   MAPE: {mape:.1f}%")
print(f"   Trend Accuracy: {trend_accuracy:.1f}%")
print(f"\n📦 Files Ready for Download:")
print(f"   1. lstm_trend_forecaster.h5 (model weights)")
print(f"   2. lstm_scaler_X.pkl (input scaler)")
print(f"   3. lstm_scaler_y.pkl (output scaler)")
print(f"   4. lstm_metadata.json (performance metrics)")
print(f"\n📝 Next Steps:")
print(f"   1. Download all 4 files")
print(f"   2. Place in: ml_models/trained/")
print(f"   3. Model will be auto-loaded by LSTMTrendPredictor")
print(f"\n✅ Ready for production use!")
print(f"{'='*70}")
</VSCode.Cell>
````